## Análisis estadístico de la base de datos de batallas

In [ ]:
import pandas as pd
import numpy as np

batallas = pd.read_csv("../data/csvs/batallas_detallado.csv", delimiter=',', encoding='utf-8')
batallas.head()

In [ ]:
batallas.dtypes

In [ ]:
batallas.info()

In [ ]:
n_filas, n_columnas = batallas.shape
print(f"Número de turnos: {n_filas}")
print(f"Número de variables: {n_columnas}")


In [ ]:
batallas['Id_Batalla'].nunique()


### Turnos medios por batalla

In [ ]:
turnos_por_batalla = (
    batallas
    .groupby("Id_Batalla")["Turno"]
    .max()
)

turnos_por_batalla.mean()
turnos_por_batalla.describe()

Distribución de turnos por batalla

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))

sns.histplot(turnos_por_batalla, bins=40, kde=True)

plt.title("Distribución del número de turnos por batalla")
plt.xlabel("Número de turnos")
plt.ylabel("Frecuencia")

plt.tight_layout()
plt.show()


In [ ]:
turnos_por_batalla.nsmallest(5)


In [ ]:
turnos_por_batalla.nlargest(5)


### Pokemon más usados


In [ ]:
import pandas as pd

j1 = batallas[['Id_Batalla', 'Jugador1', 'Activo_Jugador1']].copy()
j1.columns = ['Id_Batalla', 'Jugador', 'Pokemon']

j2 = batallas[['Id_Batalla', 'Jugador2', 'Activo_Jugador2']].copy()
j2.columns = ['Id_Batalla', 'Jugador', 'Pokemon']

activos = pd.concat([j1, j2], ignore_index=True)
activos_unicos = activos.drop_duplicates(
    subset=['Id_Batalla', 'Jugador', 'Pokemon']
)
top_pokemon = (
    activos_unicos['Pokemon']
    .value_counts()
    .head(15)
)
top_pokemon_df = top_pokemon.reset_index()
top_pokemon_df.columns = ['Pokemon', 'Num_Batallas']


In [ ]:
import matplotlib.pyplot as plt

# Ordenar por número de batallas
top_pokemon_df = top_pokemon_df.sort_values(
    by='Num_Batallas',
    ascending=True
)

# Crear la figura
plt.figure(figsize=(10, 6))

# Gráfica de barras horizontales
plt.barh(
    top_pokemon_df['Pokemon'],
    top_pokemon_df['Num_Batallas']
)

# Etiquetas y título
plt.xlabel('Número de batallas en las que aparece')
plt.ylabel('Pokémon')
plt.title('Top 15 Pokémon más usados (uso por batalla)')
plt.tight_layout()

plt.show()


In [ ]:
# Total de usos únicos de Pokémon (todas las batallas)
total_usos = activos_unicos.shape[0]
# Añadir columna de porcentaje de uso
top_pokemon_df['Porcentaje_Uso'] = (
    top_pokemon_df['Num_Batallas'] / total_usos * 100
)
top_pokemon_df.sort_values(
    by='Porcentaje_Uso',
    ascending=False
)


In [ ]:
# --- Pokémon activos del Jugador 1 ---
j1 = batallas[['Id_Batalla', 'Jugador1', 'Activo_Jugador1', 'Ganador']].copy()
j1.columns = ['Id_Batalla', 'Jugador', 'Pokemon', 'Ganador']
j1['Rol'] = 'Jugador1'

# --- Pokémon activos del Jugador 2 ---
j2 = batallas[['Id_Batalla', 'Jugador2', 'Activo_Jugador2', 'Ganador']].copy()
j2.columns = ['Id_Batalla', 'Jugador', 'Pokemon', 'Ganador']
j2['Rol'] = 'Jugador2'

# --- Unir ambos jugadores ---
activos = pd.concat([j1, j2], ignore_index=True)
# Cada Pokémon cuenta una vez por jugador y batalla
activos = activos.drop_duplicates(
    subset=['Id_Batalla', 'Rol', 'Pokemon']
)
# 1 si el Pokémon pertenece al jugador ganador de la batalla
activos['Victoria'] = (activos['Rol'] == activos['Ganador']).astype(int)
activos['Victoria'].value_counts()


In [ ]:
pokemon_stats = (
    activos
    .groupby('Pokemon')
    .agg(
        Num_Batallas=('Id_Batalla', 'count'),
        Victorias=('Victoria', 'sum'),
        Winrate=('Victoria', 'mean')
    )
    .reset_index()
)

# Winrate en porcentaje
pokemon_stats['Winrate'] *= 100

pokemon_top15 = pokemon_stats.merge(
    top_pokemon_df[['Pokemon']],
    on='Pokemon',
    how='inner'
).sort_values(
    by='Num_Batallas',
    ascending=False
)


In [ ]:
from scipy.stats import norm
import numpy as np

def wilson_ci(k, n, alpha=0.05):
    z = norm.ppf(1 - alpha / 2)
    p_hat = k / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(
        (p_hat * (1 - p_hat) + z**2 / (4 * n)) / n
    ) / denom
    return center - margin, center + margin
# Calcular IC
ci = pokemon_top15.apply(
    lambda r: wilson_ci(r['Victorias'], r['Num_Batallas']),
    axis=1
)

pokemon_top15['IC_inf'] = [c[0] * 100 for c in ci]
pokemon_top15['IC_sup'] = [c[1] * 100 for c in ci]


In [ ]:
pokemon_top15

In [ ]:
import matplotlib.pyplot as plt

# Ordenar por winrate
pokemon_plot = pokemon_top15.sort_values('Winrate')

# Crear figura
plt.figure(figsize=(8, 6))

# Posiciones en el eje Y
y_pos = range(len(pokemon_plot))

# Dibujar intervalos de confianza
plt.hlines(
    y=y_pos,
    xmin=pokemon_plot['IC_inf'],
    xmax=pokemon_plot['IC_sup'],
    linewidth=2
)

# Dibujar puntos del winrate
plt.plot(
    pokemon_plot['Winrate'],
    y_pos,
    'o'
)

# Línea de referencia en 50 %
plt.axvline(
    x=50,
    linestyle='--',
    linewidth=1
)

# Etiquetas
plt.yticks(
    y_pos,
    pokemon_plot['Pokemon']
)

plt.xlabel('Winrate (%)')
plt.title('Winrate e intervalos de confianza (95 %) – Top 15 Pokémon')

# Ajuste de márgenes
plt.tight_layout()

plt.show()


In [ ]:
from scipy.stats import binomtest

# Aplicar contraste binomial H0: p = 0.5
pokemon_top15['p_value'] = pokemon_top15.apply(
    lambda row: binomtest(
        k=row['Victorias'],
        n=row['Num_Batallas'],
        p=0.5,
        alternative='two-sided'
    ).pvalue,
    axis=1
)
# Decisión del contraste
pokemon_top15['Significativo'] = pokemon_top15['p_value'] < 0.05
tabla_test = pokemon_top15[
    ['Pokemon', 'Num_Batallas', 'Victorias', 'Winrate', 'p_value', 'Significativo']
].sort_values('p_value')

tabla_test


### Análisis junto con características de los Pokemon utilizados

In [ ]:
j1 = batallas[
    ['Id_Batalla', 'Activo_Jugador1',
     'velocidad_activo_j1', 'ataque_activo_j1',
     'ataque_esp_activo_j1', 'defensa_activo_j1',
     'defensa_esp_activo_j1']
].copy()

j1.columns = [
    'Id_Batalla', 'Pokemon',
    'velocidad', 'ataque',
    'ataque_especial', 'defensa',
    'defensa_especial'
]

# Añadir Jugador DESPUÉS del renombrado
j1['Jugador'] = 'Jugador1'


j2 = batallas[
    ['Id_Batalla', 'Activo_Jugador2',
     'velocidad_activo_j2', 'ataque_activo_j2',
     'ataque_esp_activo_j2', 'defensa_activo_j2',
     'defensa_esp_activo_j2']
].copy()
j2.columns = [
    'Id_Batalla', 'Pokemon',
    'velocidad', 'ataque',
    'ataque_especial', 'defensa',
    'defensa_especial'
]

j2['Jugador'] = 'Jugador2'

activos = pd.concat([j1, j2], ignore_index=True)

# Un Pokémon cuenta una vez por batalla y jugador
activos = activos.drop_duplicates(
    subset=['Id_Batalla', 'Jugador', 'Pokemon']
)


In [ ]:
stats_media = (
    activos
    .groupby(['Id_Batalla', 'Jugador'])
    .mean(numeric_only=True)
    .reset_index()
)


In [ ]:
stats_wide = stats_media.pivot(
    index='Id_Batalla',
    columns='Jugador'
)

stats_wide.columns = [
    f'{stat}_{jugador}'
    for stat, jugador in stats_wide.columns
]

stats_wide = stats_wide.reset_index()

stats_wide = stats_wide.merge(
    batallas[['Id_Batalla', 'Ganador']].drop_duplicates(),
    on='Id_Batalla',
    how='left'
)


In [ ]:
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt

def analizar_stat(df, stat):

    delta = (
        (df['Ganador'] == 'Jugador1')
        * (df[f'{stat}_Jugador1'] - df[f'{stat}_Jugador2'])
        +
        (df['Ganador'] == 'Jugador2')
        * (df[f'{stat}_Jugador2'] - df[f'{stat}_Jugador1'])
    )

    stat_w, p_value = wilcoxon(delta, alternative='greater')

    plt.figure(figsize=(7, 4))
    plt.boxplot(delta, vert=False, showfliers=False)
    plt.axvline(0, linestyle='--')
    plt.xlabel(f'{stat.replace("_", " ").capitalize()} media (ganador − perdedor)')
    plt.title(f'Diferencia de {stat.replace("_", " ")} media por batalla')
    plt.tight_layout()
    plt.show()

    return p_value


In [ ]:
stats = [
    'velocidad',
    'ataque',
    'ataque_especial',
    'defensa',
    'defensa_especial'
]

resultados = {s: analizar_stat(stats_wide, s) for s in stats}

resultados


Ahora comparamos usando el máximo de las stats en lugar de con la media.

In [ ]:
def construir_ganador_perdedor(df, stat):
    ganador = (
        (df['Ganador'] == 'Jugador1') * df[f'{stat}_Jugador1']
        +
        (df['Ganador'] == 'Jugador2') * df[f'{stat}_Jugador2']
    )

    perdedor = (
        (df['Ganador'] == 'Jugador1') * df[f'{stat}_Jugador2']
        +
        (df['Ganador'] == 'Jugador2') * df[f'{stat}_Jugador1']
    )

    return ganador, perdedor


In [ ]:
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

def comparar_distribuciones(df, stat):
    ganador, perdedor = construir_ganador_perdedor(df, stat)

    # Test Mann–Whitney U
    u_stat, p_value = mannwhitneyu(
        ganador,
        perdedor,
        alternative='two-sided'
    )

    # Gráfico (boxplot comparativo)
    plt.figure(figsize=(6, 4))
    plt.boxplot(
        [ganador, perdedor],
        labels=['Ganador', 'Perdedor'],
        showfliers=False
    )
    plt.ylabel(stat.replace('_', ' ').capitalize())
    plt.title(f'Distribución de {stat} (ganador vs perdedor)')
    plt.tight_layout()
    plt.show()

    return p_value


In [ ]:
stats = [
    'velocidad',
    'ataque',
    'ataque_especial',
    'defensa',
    'defensa_especial'
]

resultados_dist = {
    s: comparar_distribuciones(stats_wide, s)
    for s in stats
}

resultados_dist


In [ ]:
batallas.info()